# Multi-Query Decomposition — Split, Search, Merge

**Problem:** Complex queries have multiple aspects that a single search can't cover.
- "Drug interactions between blood thinners and painkillers"
- This needs info about: blood thinners + painkillers + their interactions
- A single embedding can't represent all three aspects well

**Solution:** Decompose into sub-queries, search each, merge with RRF.

**Pipeline:** Complex Query → LLM Decomposes → N Searches → RRF Merge → Results

In [ ]:
# Simulated decomposition
complex_queries = {
    "Drug interactions between blood thinners and painkillers": [
        "blood thinner anticoagulant medications warfarin",
        "painkiller NSAID ibuprofen aspirin analgesic",
        "drug interactions anticoagulants NSAIDs bleeding risk",
    ],
    "Tall art deco buildings built in New York in the 1930s": [
        "art deco architecture style buildings",
        "skyscrapers in New York City Manhattan",
        "buildings constructed completed in the 1930s",
    ],
    "What happens if Taiwan Semiconductor shuts down?": [
        "Taiwan Semiconductor TSMC operations production",
        "TSMC shutdown disruption supply chain impact",
        "companies dependent on TSMC chips Apple NVIDIA",
    ],
}

for query, subs in complex_queries.items():
    print(f"Original: \"{query}\"")
    print(f"Decomposed into {len(subs)} sub-queries:")
    for i, sq in enumerate(subs, 1):
        print(f"  {i}. \"{sq}\"")
    print()

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Healthcare documents
docs = [
    "Warfarin is an anticoagulant (blood thinner) used to prevent blood clots. It interacts with many drugs including NSAIDs.",
    "Ibuprofen 400mg is an NSAID used for pain relief. Contraindicated in patients on anticoagulants due to bleeding risk.",
    "Drug interaction alert: Aspirin combined with warfarin significantly increases gastrointestinal bleeding risk.",
    "Acetaminophen 500mg is a pain reliever. Unlike NSAIDs, it does NOT increase bleeding risk. Safe for patients on warfarin.",
    "Metformin 500mg is the first-line treatment for type 2 diabetes mellitus.",
]

doc_embeddings = model.encode(docs)

In [ ]:
# Compare: single query vs multi-query search
query = "Drug interactions between blood thinners and painkillers"
sub_queries = complex_queries[query]

# Single query search
query_emb = model.encode(query)
single_scores = [cosine_sim(query_emb, d) for d in doc_embeddings]

print("=== Single Query Search ===")
print(f"Query: \"{query}\"\n")
ranked = sorted(enumerate(single_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  #{rank} ({score:.4f}): {docs[idx][:80]}...")
print()

In [ ]:
# Multi-query search with RRF fusion
print("=== Multi-Query Search (3 sub-queries + RRF) ===")

k = 60  # RRF constant
rrf_scores = {i: 0.0 for i in range(len(docs))}

for sq in sub_queries:
    sq_emb = model.encode(sq)
    sq_scores = [cosine_sim(sq_emb, d) for d in doc_embeddings]
    sq_ranked = sorted(enumerate(sq_scores), key=lambda x: x[1], reverse=True)
    
    print(f"\n  Sub-query: \"{sq}\"")
    for rank, (idx, score) in enumerate(sq_ranked[:3], 1):
        rrf_scores[idx] += 1 / (k + rank)
        print(f"    #{rank} ({score:.4f}): Doc {idx}")

print("\n=== Final RRF Ranking ===")
rrf_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(rrf_ranked, 1):
    print(f"  #{rank} (RRF={score:.6f}): {docs[idx][:80]}...")

## Why Multi-Query Beats Single Query

A single embedding for "Drug interactions between blood thinners and painkillers"
tries to represent THREE concepts in one 384-dim vector. It's a compromise.

Three separate embeddings — one per sub-query — each nail their specific aspect.
RRF then rewards documents that score well across MULTIPLE sub-queries.

## Key Takeaways

1. **Complex queries have multiple information needs** — one embedding can't cover all
2. **Decomposition + RRF** finds documents that single-query search misses
3. **LLM does the decomposition** — it understands query structure better than rules
4. **Trade-off:** N sub-queries = N searches = N× latency. Worth it for complex queries
5. **Best for:** Multi-constraint queries, comparison queries, multi-hop questions